In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q unsloth

In [3]:
import torch
from unsloth import FastLanguageModel

MODEL_ID = "IITM-2026T3-DSAI-GROUP2/NCERT-AI-Tutor-Qwen3.5-4B"

model, processor = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    dtype=None,
    load_in_4bit=False,
)

FastLanguageModel.for_inference(model)

print("Model and LoRA adapter loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.6: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with r

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: the model has bfloat16 layers on GPU 0 (Tesla T4), which has no native bfloat16, so they run slowly. Load with dtype = torch.float16, or keep the model on bfloat16 GPUs with CUDA_VISIBLE_DEVICES.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Model and LoRA adapter loaded successfully!


In [5]:
messages = [
    {
        "role": "system",
        "content": """
You are a friendly AI tutor for school students.

Rules:
1. Speak directly to the student.
2. Use simple, age-appropriate English.
3. Explain concepts step by step.
4. Never display internal reasoning, analysis, or planning.
5. Start directly with the explanation.
6. Use one real-life example.
7. End with one question to check understanding.
8. Keep the answer clear and concise.
"""
    },
    {
        "role": "user",
        "content": "I am a Class 7 student. Explain photosynthesis with a simple real-life example."
    }
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.7,
        top_p=0.8,
        do_sample=True,
        repetition_penalty=1.05,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("AI TUTOR RESPONSE:\n")
print(answer)

AI TUTOR RESPONSE:

Photosynthesis is how plants make their own food using sunlight. For example, just like you need sunlight to grow, plants use sunlight to turn water and carbon dioxide into sugar (food) and oxygen.



In [6]:

import torch
from IPython.display import display, Markdown

# 12 test questions across different subjects and skills
test_questions = [
    ("Science - Class 7",
     "Explain photosynthesis with a simple real-life example."),

    ("Maths - Class 7",
     "A shopkeeper sells a pen for Rs. 20 and earns 25% profit. Find its cost price. Explain step by step."),

    ("Science - Class 8",
     "Explain Newton's third law of motion with two everyday examples."),

    ("Maths - Conceptual understanding",
     "I think 1/2 + 1/3 = 2/5. Am I correct? Do not directly give me the answer. Help me understand my mistake using hints."),

    ("Biology - Class 9",
     "What is the difference between plant cells and animal cells? Explain in simple English."),

    ("Physics - Class 8",
     "Why do we see lightning before hearing thunder? Explain the scientific reason."),

    ("Chemistry - Class 7",
     "Explain the difference between physical change and chemical change with three examples."),

    ("English - Class 6",
     "Explain the difference between 'their', 'there', and 'they're' with examples."),

    ("Reasoning",
     "A farmer has 17 sheep. All but 9 run away. How many sheep are left? Explain your reasoning."),

    ("Teaching ability",
     "I am unable to understand fractions. Teach me fractions as if I am a beginner, using a pizza example."),

    ("Follow-up handling",
     "I still don't understand why plants need sunlight. Explain it again using simpler words and a different example."),

    ("Critical thinking",
     "If plants release oxygen during photosynthesis, does that mean plants do not need oxygen? Explain your answer.")
]

# Get tokenizer
tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

model.eval()
model.config.use_cache = True

results = []

for i, (category, question) in enumerate(test_questions, 1):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a friendly AI tutor for school students. "
                "Use simple, age-appropriate English. "
                "Explain concepts clearly and step by step. "
                "Never show internal reasoning or thinking process. "
                "For mistakes, guide students with hints instead of "
                "immediately revealing the answer. "
                "Use examples wherever helpful."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    results.append({
        "category": category,
        "question": question,
        "answer": answer
    })

    print("\n" + "=" * 85)
    print(f"TEST {i}/12: {category}")
    print("=" * 85)
    print("QUESTION:", question)
    print("\nAI TUTOR RESPONSE:\n", answer)

    del inputs, outputs, new_tokens
    torch.cuda.empty_cache()

print("\n" + "=" * 85)
print("ALL 12 TESTS COMPLETED")
print("=" * 85)



TEST 1/12: Science - Class 7
QUESTION: Explain photosynthesis with a simple real-life example.

AI TUTOR RESPONSE:
 Photosynthesis is like a plant's 'kitchen'. Plants take in sunlight (the stove), water and carbon dioxide (the ingredients), and cook them to produce food (glucose) and oxygen (a byproduct we breathe).

TEST 2/12: Maths - Class 7
QUESTION: A shopkeeper sells a pen for Rs. 20 and earns 25% profit. Find its cost price. Explain step by step.

AI TUTOR RESPONSE:
 Step 1: Profit = 25% of Cost Price (CP). Step 2: Let CP be x. Then, 0.25x is the profit. Step 3: Selling Price (SP) = CP + Profit = x + 0.25x = 1.25x. Step 4: Given SP is 20, so 1.25x = 20. Step 5: Solving for x, x = 20 / 1.25 = 16. The cost price is Rs. 16.

TEST 3/12: Science - Class 8
QUESTION: Explain Newton's third law of motion with two everyday examples.

AI TUTOR RESPONSE:
 Newton's Third Law states: 'For every action, there is an equal and opposite reaction.' Examples: 1) When you jump, your feet push down 

In [7]:

import torch
import time
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# 1. TEST DATASET: 15 QUESTIONS
# ==========================================

test_questions = [

    # IN-DOMAIN: EDUCATIONAL (8)
    ("Educational", "Science - Class 7",
     "Explain photosynthesis with a simple real-life example."),

    ("Educational", "Maths - Class 7",
     "A shopkeeper sells a pen for Rs. 20 and earns 25% profit. Find its cost price step by step."),

    ("Educational", "Maths - Fractions",
     "I think 1/2 + 1/3 = 2/5. Am I correct? Guide me using hints. Do not directly give the answer."),

    ("Educational", "Physics - Class 8",
     "Explain Newton's third law with two everyday examples."),

    ("Educational", "Biology - Class 9",
     "What is the difference between plant cells and animal cells?"),

    ("Educational", "Chemistry - Class 7",
     "Explain physical and chemical changes with three examples each."),

    ("Educational", "English - Class 6",
     "Explain the difference between their, there and they're with examples."),

    ("Educational", "Reasoning",
     "A farmer has 17 sheep. All but 9 run away. How many sheep are left? Explain."),

    # OFF-TOPIC / GENERAL: (7)
    ("General", "Movies",
     "What is the central message of the Indian movie 3 Idiots? Explain in simple English."),

    ("General", "Politics",
     "Explain the difference between democracy and dictatorship in a neutral and unbiased way."),

    ("General", "Relationships",
     "My best friend has stopped talking to me. What should I do? Give me some practical advice."),

    ("General", "Computer Science",
     "Explain binary search in simple language with an example. Why must the array be sorted?"),

    ("General", "Technology",
     "What is the difference between Artificial Intelligence, Machine Learning and Deep Learning?"),

    ("General", "Cooking",
     "Why does adding salt to water change its boiling point? Explain simply."),

    ("General", "General Knowledge",
     "Why does the sky appear blue during the day? Explain the science behind it.")
]


# ==========================================
# 2. MODEL CONFIGURATION
# ==========================================

tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor

model.eval()
model.config.use_cache = True

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

results = []


# ==========================================
# 3. RUN ALL TESTS
# ==========================================

for i, (group, category, question) in enumerate(test_questions, 1):

    messages = [
        {
            "role": "system",
            "content": """
You are a friendly educational AI tutor.

Your primary purpose is helping students learn school subjects.

Instructions:
1. Explain educational topics in simple, age-appropriate language.
2. Give step-by-step explanations when needed.
3. Never provide incorrect formulas or facts.
4. Guide students using hints when they make mistakes.
5. You may also answer harmless general questions about movies,
   politics, relationships, computer science and daily life.
6. Be neutral when discussing politics.
7. Be empathetic and respectful when discussing relationships.
8. If you are uncertain, say so instead of inventing facts.
9. Do not show internal thinking or hidden reasoning.
10. Keep answers relevant to the question.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    # Measure generation latency
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_time = time.perf_counter()

    latency = end_time - start_time

    # Extract generated tokens only
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    token_count = len(generated_tokens)

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    tokens_per_second = (
        token_count / latency if latency > 0 else 0
    )

    word_count = len(answer.split())

    results.append({
        "Test": i,
        "Group": group,
        "Category": category,
        "Question": question,
        "Answer": answer,
        "Latency_sec": round(latency, 2),
        "Generated_tokens": token_count,
        "Tokens_per_sec": round(tokens_per_second, 2),
        "Word_count": word_count,
        "Empty_response": len(answer) == 0,
        "Manual_quality_score_1_to_5": None
    })

    print("\n" + "=" * 90)
    print(f"TEST {i}/15 | {group} | {category}")
    print("=" * 90)
    print("QUESTION:", question)
    print("\nAI RESPONSE:\n", answer)
    print(f"\nLatency: {latency:.2f} seconds")
    print(f"Generated tokens: {token_count}")
    print(f"Tokens/second: {tokens_per_second:.2f}")

    del inputs, outputs, generated_tokens
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ==========================================
# 4. PERFORMANCE METRICS
# ==========================================

df = pd.DataFrame(results)

print("\n\n")
print("=" * 90)
print("AI TUTOR PERFORMANCE REPORT")
print("=" * 90)

print("\nOVERALL METRICS")

print("Total questions:", len(df))
print("Successful non-empty responses:", (~df["Empty_response"]).sum())
print("Empty responses:", df["Empty_response"].sum())

print("Average latency:",
      round(df["Latency_sec"].mean(), 2), "seconds")

print("Average generated tokens:",
      round(df["Generated_tokens"].mean(), 2))

print("Average generation speed:",
      round(df["Tokens_per_sec"].mean(), 2), "tokens/sec")

print("Average response words:",
      round(df["Word_count"].mean(), 2))

print("\nGROUP-WISE METRICS")

group_metrics = df.groupby("Group").agg(
    Questions=("Test", "count"),
    Avg_Latency=("Latency_sec", "mean"),
    Avg_Tokens=("Generated_tokens", "mean"),
    Avg_Tokens_Per_Sec=("Tokens_per_sec", "mean"),
    Avg_Words=("Word_count", "mean"),
    Empty_Responses=("Empty_response", "sum")
).round(2)

display(group_metrics)

print("\nINDIVIDUAL TEST METRICS")

display(
    df[[
        "Test", "Group", "Category",
        "Latency_sec", "Generated_tokens",
        "Tokens_per_sec", "Word_count"
    ]]
)

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print("\nPeak GPU memory allocated:",
          round(peak_memory, 2), "GB")


# ==========================================
# 5. SAVE COMPLETE REPORT
# ==========================================

df.to_csv(
    "/kaggle/working/ai_tutor_15_test_results.csv",
    index=False
)

df.to_json(
    "/kaggle/working/ai_tutor_15_test_results.json",
    orient="records",
    indent=4
)

print("\nReports saved:")
print("/kaggle/working/ai_tutor_15_test_results.csv")
print("/kaggle/working/ai_tutor_15_test_results.json")

print("\nALL 15 TESTS COMPLETED SUCCESSFULLY!")



TEST 1/15 | Educational | Science - Class 7
QUESTION: Explain photosynthesis with a simple real-life example.

AI RESPONSE:
 Photosynthesis is like a plant's kitchen. Plants take sunlight (the stove), water and carbon dioxide from the air (ingredients), and cook them into sugar (food) for themselves.

Latency: 2.26 seconds
Generated tokens: 40
Tokens/second: 17.72

TEST 2/15 | Educational | Maths - Class 7
QUESTION: A shopkeeper sells a pen for Rs. 20 and earns 25% profit. Find its cost price step by step.

AI RESPONSE:
 Step 1: Profit = Selling Price - Cost Price. Step 2: 25% of Cost Price = 20 - Cost Price. Step 3: 0.25 * Cost Price + Cost Price = 20. Step 4: 1.25 * Cost Price = 20. Step 5: Cost Price = 20 / 1.25 = Rs. 16.

Latency: 4.88 seconds
Generated tokens: 94
Tokens/second: 19.28

TEST 3/15 | Educational | Maths - Fractions
QUESTION: I think 1/2 + 1/3 = 2/5. Am I correct? Guide me using hints. Do not directly give the answer.

AI RESPONSE:
 That's a good start! But remember, 

,Questions,Avg_Latency,Avg_Tokens,Avg_Tokens_Per_Sec,Avg_Words,Empty_Responses
Group,,,,,,
Educational,8,3.14,58.50,18.48,35.5,0
General,7,3.00,54.86,18.21,44.0,0



INDIVIDUAL TEST METRICS


,Test,Group,Category,Latency_sec,Generated_tokens,Tokens_per_sec,Word_count
0,1,Educational,Science - Class 7,2.26,40,17.72,27
1,2,Educational,Maths - Class 7,4.88,94,19.28,50
2,3,Educational,Maths - Fractions,2.13,38,17.84,28
3,4,Educational,Physics - Class 8,3.87,74,19.13,55
4,5,Educational,Biology - Class 9,2.21,40,18.14,26
5,6,Educational,Chemistry - Class 7,3.94,75,19.02,37
6,7,Educational,English - Class 6,3.37,63,18.72,32
7,8,Educational,Reasoning,2.45,44,17.97,29
8,9,General,Movies,2.43,44,18.08,36
9,10,General,Politics,2.34,42,17.94,35



Peak GPU memory allocated: 8.7 GB

Reports saved:
/kaggle/working/ai_tutor_15_test_results.csv
/kaggle/working/ai_tutor_15_test_results.json

ALL 15 TESTS COMPLETED SUCCESSFULLY!
